# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [ ]:
#!pip install -qU ragas==0.2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/1

In [ ]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/xtallet/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/xtallet/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [4]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [5]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [6]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [7]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/home/xtallet/AIMakerSpace/AIE7/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/home/xtallet/AIMakerSpace/AIE7/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/home/xtallet/AIMakerSpace/AIE7/07_Synthetic_Data_Generation_and_LangSmith/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [8]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [9]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [10]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '623c04'. Skipping!
Property 'summary' already exists in node '06e037'. Skipping!
Property 'summary' already exists in node 'c857c7'. Skipping!
Property 'summary' already exists in node 'e7929c'. Skipping!
Property 'summary' already exists in node '1c476e'. Skipping!
Property 'summary' already exists in node '40a5e3'. Skipping!
Property 'summary' already exists in node '27eb80'. Skipping!
Property 'summary' already exists in node '88c33d'. Skipping!
Property 'summary' already exists in node '03c01e'. Skipping!
Property 'summary' already exists in node '4d6f5f'. Skipping!
Property 'summary' already exists in node 'f20cf5'. Skipping!
Property 'summary' already exists in node 'd75819'. Skipping!
Property 'summary' already exists in node 'e138c2'. Skipping!
Property 'summary' already exists in node '003b58'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/43 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '623c04'. Skipping!
Property 'summary_embedding' already exists in node '06e037'. Skipping!
Property 'summary_embedding' already exists in node 'f20cf5'. Skipping!
Property 'summary_embedding' already exists in node 'e7929c'. Skipping!
Property 'summary_embedding' already exists in node '88c33d'. Skipping!
Property 'summary_embedding' already exists in node '40a5e3'. Skipping!
Property 'summary_embedding' already exists in node 'c857c7'. Skipping!
Property 'summary_embedding' already exists in node 'd75819'. Skipping!
Property 'summary_embedding' already exists in node '1c476e'. Skipping!
Property 'summary_embedding' already exists in node '003b58'. Skipping!
Property 'summary_embedding' already exists in node 'e138c2'. Skipping!
Property 'summary_embedding' already exists in node '27eb80'. Skipping!
Property 'summary_embedding' already exists in node '03c01e'. Skipping!
Property 'summary_embedding' already exists in node '4d6f5f'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 40, relationships: 478)

#### Screenshots 

These are all the LangSmith traces generated during the transformation process.
In this screenshot I am going to show you one of them.

<img src="screenshots/transformations_trace.png" alt="Loan Synthetic Data" width="1200"/>

We can save and load our knowledge graphs as follows.

In [11]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 40, relationships: 478)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [12]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [14]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5), # 🏗️ XTALLET Notes - Generates simple and direct questions
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25), # 🏗️ XTALLET Notes - Generates more complex and abstract questions
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25), # 🏗️ XTALLET Notes - Generates specific and complex questions
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### ✅ Answer:

- SingleHopSpecificQuerySynthesizer :<br>
  It generates simple and direct questions that can be answered by looking at a single piece of information in the text.<br>
  It makes easy and direct questions.<br>
  These questions will represent the 50% of the total list of queries.

- MultiHopAbstractQuerySynthesizer :<br>
  Generates more complex and general questions that require combining information from different parts of the text and thinking more abstractly.<br>
  It makes broad and challenging questions.<br>
  These questions will represent the 25% of the total list of queries.

- MultiHopSpecificQuerySynthesizer :<br>
  Generates specific but complex questions that require searching and connecting information from multiple parts of the text to give a concrete answer.<br>
  It makes detailed questions that require gathering information from different parts of the text.<br>
  These questions will represent the 25% of the total list of queries.

Finally, we can use our `TestSetGenerator` to generate our testset!

In [15]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,Academic Years what are they?,"[Chapter 1 Academic Years, Academic Calendars,...","Chapter 1 Academic Years, Academic Calendars, ...",single_hop_specifc_query_synthesizer
1,Could you please explain the significance of 3...,[Regulatory Citations Academic year minimums: ...,Regulatory citations indicate that 34 CFR 668....,single_hop_specifc_query_synthesizer
2,What is Volume 8 in the context of academic te...,[Inclusion of Clinical Work in a Standard Term...,"In the provided context, Volume 8 refers to Ch...",single_hop_specifc_query_synthesizer
3,Is the Federal Work-Study (FWS) program subjec...,[Non-Term Characteristics A program that measu...,"No, the payment period is applicable to all Ti...",single_hop_specifc_query_synthesizer
4,Is the Pell Grant still available if a student...,[both the credit or clock hours and the weeks ...,The Pell Grant amount that a student is eligib...,single_hop_specifc_query_synthesizer
5,"How do the regulatory citations, specifically ...","[<1-hop>\n\nChapter 1 Academic Years, Academic...",The regulatory citations 34 CFR 668.3(a) and 3...,multi_hop_abstract_query_synthesizer
6,so if clinical work is outside the term and al...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Inclusion of clinical work in a standard term ...,multi_hop_abstract_query_synthesizer
7,How do the regulations governing Title IV prog...,[<1-hop>\n\nboth the credit or clock hours and...,The regulations governing Title IV programs sp...,multi_hop_abstract_query_synthesizer
8,How do the disbursement timing rules outlined ...,[<1-hop>\n\nboth the credit or clock hours and...,The disbursement timing rules detailed in Appe...,multi_hop_specific_query_synthesizer
9,How do Volume 2 and Volume 8 relate to the inc...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Volume 2 discusses the requirements for defini...,multi_hop_specific_query_synthesizer


#### Screenshots 

These are all the LangSmith traces generated during generator process.
In this screenshot I am going to show you one of them.

<img src="screenshots/ragas_trace.png" alt="Loan Synthetic Data" width="1200"/>

### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [16]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '05a7ad'. Skipping!
Property 'summary' already exists in node 'd939c3'. Skipping!
Property 'summary' already exists in node '719616'. Skipping!
Property 'summary' already exists in node '47c7d9'. Skipping!
Property 'summary' already exists in node '3dfbfc'. Skipping!
Property 'summary' already exists in node '643758'. Skipping!
Property 'summary' already exists in node '81980a'. Skipping!
Property 'summary' already exists in node 'c95615'. Skipping!
Property 'summary' already exists in node '1ea70d'. Skipping!
Property 'summary' already exists in node 'd1a824'. Skipping!
Property 'summary' already exists in node '9059da'. Skipping!
Property 'summary' already exists in node 'e5c264'. Skipping!
Property 'summary' already exists in node '62ad48'. Skipping!
Property 'summary' already exists in node '285337'. Skipping!


...

Property 'summary_embedding' already exists in node '05a7ad'. Skipping!
Property 'summary_embedding' already exists in node 'e5c264'. Skipping!
Property 'summary_embedding' already exists in node 'd939c3'. Skipping!
Property 'summary_embedding' already exists in node '62ad48'. Skipping!
Property 'summary_embedding' already exists in node '81980a'. Skipping!
Property 'summary_embedding' already exists in node '719616'. Skipping!
Property 'summary_embedding' already exists in node '643758'. Skipping!
Property 'summary_embedding' already exists in node '47c7d9'. Skipping!
Property 'summary_embedding' already exists in node 'd1a824'. Skipping!
Property 'summary_embedding' already exists in node '3dfbfc'. Skipping!
Property 'summary_embedding' already exists in node '1ea70d'. Skipping!
Property 'summary_embedding' already exists in node 'c95615'. Skipping!
Property 'summary_embedding' already exists in node '9059da'. Skipping!
Property 'summary_embedding' already exists in node '285337'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

#### Screenshots 

These are all the LangSmith traces generated during TestSetGeneration process.
In this screenshot I am going to show you one of them.

<img src="screenshots/ragas_trace_tessetgenerator.png" alt="Loan Synthetic Data" width="1200"/>

In [17]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,How does the Title IV regulation define the ac...,"[Chapter 1 Academic Years, Academic Calendars,...","For Title IV purposes, the academic year is de...",single_hop_specifc_query_synthesizer
1,What is 34 CFR 668.3(a) about in terms of acad...,[Regulatory Citations Academic year minimums: ...,Regulatory Citations Academic year minimums ar...,single_hop_specifc_query_synthesizer
2,Can you explane what Chapter 3 is in the conte...,[Inclusion of Clinical Work in a Standard Term...,Inclusion of Clinical Work in a Standard Term ...,single_hop_specifc_query_synthesizer
3,Is the FWS program considered a term or non-te...,[Non-Term Characteristics A program that measu...,The FWS program is not considered a payment pe...,single_hop_specifc_query_synthesizer
4,How do credit-hour and clock-hour programs dif...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Credit-hour programs require an academic year ...,multi_hop_abstract_query_synthesizer
5,whats requried for clinikal or practicum exper...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Inclusion of clinical work in a standard term ...,multi_hop_abstract_query_synthesizer
6,How do the accelrated effects on disbursement ...,[<1-hop>\n\nboth the credit or clock hours and...,The context explains that in clock-hour or non...,multi_hop_abstract_query_synthesizer
7,How do payment scheduling and installment opti...,[<1-hop>\n\nboth the credit or clock hours and...,Payment scheduling and installment options dir...,multi_hop_abstract_query_synthesizer
8,which volumes are in Volume 2 and Volume 8?,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The provided context does not specify the cont...,multi_hop_specific_query_synthesizer
9,How do the definitions of academic years and i...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Volume 2 specifies that an academic year must ...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [19]:
from langsmith import Client

client = Client()

dataset_name = "Loan Synthetic Data"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [20]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [21]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [22]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [23]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [24]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan RAG"
)

In [25]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [26]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [27]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [28]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [29]:
rag_chain.invoke({"question" : "What kinds of loans are available?"})

'The kinds of loans available mentioned in the context are:\n\n- Direct Subsidized Loans  \n- Direct Unsubsidized Loans  \n- Direct PLUS Loans (including student Federal PLUS Loans and parent Direct PLUS Loans)  \n- Subsidized and Unsubsidized Federal Stafford Loans (made under the FFEL Program before July 1, 2010)  \n- Federal SLS Loans (made under the FFEL Program before July 1, 2010)  \n- Federal PLUS Loans (made under the FFEL Program before July 1, 2010)  \n- Direct Consolidation Loans  \n- Federal Consolidation Loans (under the FFEL Program)\n\nAdditionally, Direct Subsidized Loans are available only to undergraduate students. Graduate or professional students are eligible only for Direct Unsubsidized Loans and Direct PLUS Loans.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [30]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [31]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
- `labeled_helpfulness_evaluator`:
- `empathy_evaluator`:

##### ✅ Answer:

- QA_EVALUATOR :<br>
  Evaluates the correctness of the model's answer. It checks wheter the generated response accurately answers the user's question based on the reference answer (the answer from the synthetic dataset).

- LABELED_HELPFULNESS_EVALUATOR :<br>
  Evaluates the helpfulness of the response. It determines if the answer is useful and supportive to the user, taking into account the correct reference answer (the answer from the synthetic dataset).

- EMPATHY_EVALUATOR :<br>
  Evaluates the empathy in the response. It checks whether the answer is empathetic and makes the user feel heard and understood.
   

## LangSmith Evaluation

In [32]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'fresh-snow-25' at:
https://smith.langchain.com/o/c21c7da9-346c-4a28-b19c-bb6d2faffe60/datasets/67ac278f-3abf-48d8-8455-2326b1a061af/compare?selectedSessions=a95c8f3f-ef2f-4c18-aa32-d72952e57e0d




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,Considering the information from Volume 8 rega...,Based on the provided context:\n\nClinical wor...,None,The policies outlined in Volume 8 specify that...,1,1,0,5.815710,8d4e6d1d-26d8-4e19-b041-e5a1e3c83dfc,3ea0585d-e166-4b1d-b637-781f7c920cef
1,How do Appendix A and Appendix B relate to dis...,"Based on the provided context, Appendix B (Vol...",None,Appendix B provides detailed guidance on disbu...,0,0,0,3.760545,1f9a1adc-48ff-4f11-bcda-487f0cb89f6b,db98dfc8-0233-4b2a-bdf7-b6478a121445
2,How do the definitions of academic years and i...,I don't know.,None,Volume 2 specifies that an academic year must ...,0,0,0,0.775730,7134f3b7-dec4-4ac9-aebe-786c5fd20a66,d3c30e73-7540-46aa-b9e5-2c787d01d772
3,which volumes are in Volume 2 and Volume 8?,"Based on the provided context, Volume 8 is tit...",None,The provided context does not specify the cont...,0,0,0,1.756533,aee1b9b4-736c-4d4a-9ad8-b0c08dd569f1,19fd90d4-1993-4bc4-9288-02dd4496b121
4,How do payment scheduling and installment opti...,"Based on the provided context, payment schedul...",None,Payment scheduling and installment options dir...,1,0,0,6.374068,efdddb41-6e93-41a3-9db8-c65e0b3cabdb,4c83affc-453d-4ec7-aaa6-e35ebb94b3f1
5,How do the accelrated effects on disbursement ...,Based on the context provided:\n\nIn clock-hou...,None,The context explains that in clock-hour or non...,1,1,0,5.473835,0487fd2f-c09e-4459-b465-8b64a8430b09,32ae2025-d971-4bbc-9849-3061721bbec7
6,whats requried for clinikal or practicum exper...,For clinical or practicum experience to be inc...,None,Inclusion of clinical work in a standard term ...,1,1,0,4.015139,cc007a85-9e06-46c0-82f4-536f7a9549c3,18b69ea1-e07d-4c65-b3ec-a0cc8fc0fe07
7,How do credit-hour and clock-hour programs dif...,Based on the provided context:\n\nFor undergra...,None,Credit-hour programs require an academic year ...,0,0,0,4.843865,de2e82cf-478a-4c71-b2a4-808fcfd0bf6d,e4a2f94f-bd84-42dd-bce5-2953aed79040
8,Is the FWS program considered a term or non-te...,I don't know.,None,The FWS program is not considered a payment pe...,0,0,0,2.010965,7fb490d4-75ce-481c-a551-415985736273,ebe4fe00-580e-40aa-90ef-658ab88baec2
9,Can you explane what Chapter 3 is in the conte...,I don't know.,None,Inclusion of Clinical Work in a Standard Term ...,0,0,0,1.056049,9ba38a84-b00b-46e7-934a-2014c68851bd,285fbb61-3508-4b69-a54a-f316f6985ae9


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [33]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [34]:
rag_documents = docs

In [35]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

##### ✅ Answer:

Modifying the chunk size can signigicantly impact on :
 - Retrieval Quality :<br> 
   Smaller chunks provide more precise and focused information   retrieval, but may miss broader context.<br>
   Larger chunks : Capture more comprehensive context but may include irrelevant information that dilutes the answer quality.

 - Context Window Efficiency :<br>
   Smaller chunks fit better within the LLM's context window, allowing more relevant chunks to be included in the prompt.<br>
   Larger chunks consume more of the context window, potentially limiting the number of different sources that can be referenced.

 - Semantic Search Accuracy :<br>
   Smaller chunks create more granular embeddings, making it easier to find highly specific information that directly answers the question.<br>
   Larger chunks may have broader semantic meaning but could be less precise for specific queries.

 - Information Completeness :<br>
   Smaller chunks might fragment important information that spans across chunk boundaries, leading to incomplete answers.<br>
   Larger chunks are more likely to contain complete information but may include unnecessary details.

 - Processing Speed :<br>
   Smaller chunks generally result in faster embedding generation and retrieval due to their size.<br>
   Larger chunks require more computational resources but may reduce the number of chunks to process.<br>

   After modify the `chunk_size` and `chunk_overlap` it would affect how the system retrieves and processes the information, potentially improving or degrading performance depending on the specific use case.

In [36]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

##### ✅ Answer:

Modifying the embedding model can significantly impact on :
 
 - Semantic Understanding Quality:<br>
   Different embedding models have varying capabilities in understanding semantic relationships, context, and nuances in text.<br>
   More advanced models (like text-embedding-3-large vs text-embedding-3-small) can capture more sophisticated semantic patterns and relationships.<br>
   Better semantic understanding leads to more accurate retrieval of relevant context for user queries.

 - Retrieval Accuracy :<br>
   Embedding quality directly affects how well the vector similarity search works.<br>
   Superior embedding models can better distinguish between similar but different concepts, reducing false positives and improving precision.<br>
   Poor embeddings may lead to retrieving irrelevant chunks or missing important context.

 - Multilingual and Domain Performance :<br>
   Different models may perform better on specific languages, domains, or types of content.<br>
   Specialized models might handle technical jargon, industry-specific terms, or cultural context better than general-purpose models.

 - Dimensionality and Representation :<br>
   Different embedding dimensions can capture more or less information.<br>
   Higher dimensionality often means richer representations but may require more computational resources.

 - Training Data and Knowledge Cutoff :<br>
   Newer models may have been trained on more recent data, better understanding current events, terminology, or concepts.<br>
   Model updates can improve performance on specific types of queries or content.

In [37]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan Data for RAG"
)

In [38]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [39]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [40]:
empathy_rag_chain.invoke({"question" : "What kinds of loans are available?"})

"Thank you for your question. Based on the information provided in the context, there are several kinds of loans available:\n\n1. **Direct Subsidized Loans** – These loans are available based on the student's financial need and can cover part of the student's cost of attendance (COA).\n\n2. **Direct Unsubsidized Loans** – These loans are available regardless of financial need and can be combined with subsidized loans to meet the student's COA.\n\n3. **Direct PLUS Loans** – These loans are available to parents of dependent undergraduate students to help pay the student's COA, assuming the parent meets eligibility requirements. They are also available to graduate and professional students.\n\nIt's also noted that if a dependent student's parent cannot get a Direct PLUS Loan, the student may be eligible for additional Direct Unsubsidized Loan amounts to help cover costs.\n\nI hope this helps clarify the types of loans available. If you have more questions or need further details, please f

Finally, we can evaluate the new chain on the same test set!

In [41]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'excellent-cheek-28' at:
https://smith.langchain.com/o/c21c7da9-346c-4a28-b19c-bb6d2faffe60/datasets/67ac278f-3abf-48d8-8455-2326b1a061af/compare?selectedSessions=61d98dac-8256-4234-9bd3-5fa692e5fba5




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,Considering the information from Volume 8 rega...,Thank you for your thoughtful question. It sou...,None,The policies outlined in Volume 8 specify that...,1,1,1,7.887762,8d4e6d1d-26d8-4e19-b041-e5a1e3c83dfc,0da8fd47-2658-483d-b8a2-2d7e38ba46d5
1,How do Appendix A and Appendix B relate to dis...,Thank you for your thoughtful question. From t...,None,Appendix B provides detailed guidance on disbu...,0,0,1,4.622206,1f9a1adc-48ff-4f11-bcda-487f0cb89f6b,8b0c6e49-d1e2-466e-b570-562d8330f67b
2,How do the definitions of academic years and i...,Thank you for your thoughtful question. From t...,None,Volume 2 specifies that an academic year must ...,0,0,1,5.913212,7134f3b7-dec4-4ac9-aebe-786c5fd20a66,e655246c-f26a-4ab1-b833-5c57a47fb604
3,which volumes are in Volume 2 and Volume 8?,Thank you for your question. Based on the cont...,None,The provided context does not specify the cont...,0,0,1,2.805967,aee1b9b4-736c-4d4a-9ad8-b0c08dd569f1,518cc3a0-1dc1-47e5-af12-af693e64b9bc
4,How do payment scheduling and installment opti...,Thank you for your thoughtful question about h...,None,Payment scheduling and installment options dir...,1,1,1,5.309599,efdddb41-6e93-41a3-9db8-c65e0b3cabdb,ad42f342-3b94-499e-9405-363c32f65ef1
5,How do the accelrated effects on disbursement ...,Thank you for your thoughtful question. It’s c...,None,The context explains that in clock-hour or non...,1,0,1,4.922457,0487fd2f-c09e-4459-b465-8b64a8430b09,de26b5c5-bd61-487a-afd3-aae8daf94003
6,whats requried for clinikal or practicum exper...,Thank you for your thoughtful question! Based ...,None,Inclusion of clinical work in a standard term ...,1,1,1,4.692234,cc007a85-9e06-46c0-82f4-536f7a9549c3,e35a357b-4f70-4c50-852b-bfb5a213432f
7,How do credit-hour and clock-hour programs dif...,Thank you for your thoughtful question. Based ...,None,Credit-hour programs require an academic year ...,1,1,1,3.658594,de2e82cf-478a-4c71-b2a4-808fcfd0bf6d,c35028b9-0a50-4103-b5a5-f60df636f0fc
8,Is the FWS program considered a term or non-te...,Thank you for your question! Based on the cont...,None,The FWS program is not considered a payment pe...,1,1,1,2.653301,7fb490d4-75ce-481c-a551-415985736273,bcd965a1-8fd4-4437-ae1a-b30824b8323e
9,Can you explane what Chapter 3 is in the conte...,Thank you for your question! I completely unde...,None,Inclusion of Clinical Work in a Standard Term ...,0,0,1,2.675091,9ba38a84-b00b-46e7-934a-2014c68851bd,22c483bc-8581-49a3-80b3-bb682602248b


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

##### ✅ Answer:

First Evaluation Config : 
 - Prompt : RAG_PROMPT
 - Embedding Model : text-embedding-3-amsll
 - Chunking : chunk_size : 500 - chunk_overlap : 50 

Second Evaluation Config : 
 - Prompt : EMPATHY_RAG_PROMPT
 - Embedding Model : text-embedding-3-large
 - Chunking : chunk_size : 1000 - chunk_overlap : 50

These are my observations on the three types of evaluations we have tested :<br>

#### Correctness
The number of answers marked as correct increased in the second evaluation.<br>
Probable reasons :<br>

- Better Embedding model improved the retrieval of relevant context, so the model had more accurate information to answer the questions.<br>
- Larger chunk size meant each retrieved chunk contained more context, reducing the chance of missing key information needed for a correct answer.<br>
- The prompt change to an empathy-focused version did not negatively affect correctness, as the model was still instructed to answer based on context.

#### Helpfulness
Helpfulness scores generally improved or remained high for correct answers.<br>
Probable reasons :<br>

 - More relevant and complete context (due to better embeddings and larger chunks) allowed the model to provide more useful and informative answers.
 - The empathy prompt may have encouraged the model to give more supportive and user-focused responses, which can be perceived as more helpful.

#### Empathy
Empathy scores increased significantly in the second evaluation.<br>
Probable reasons :<br>

- The empathy-focused prompt explicitly instructed the model to be empathetic and make the user feel heard.<br>
- As a result, the model’s responses included more empathetic language, which was recognized by the evaluator.<br>

#### Screenshots
 - This screenshot shows both evaluations comparisson
<img src="screenshots/activity3_scrn1.png" alt="Loan Synthetic Data" width="1200"/>

 - This screenshot shows the details about the Evaluation 1 :
 <img src="screenshots/activity3_scrn2.png" alt="Loan Synthetic Data" width="1200"/>

 - This screenshot shows the details about the Evaluation 2 :
 <img src="screenshots/activity3_scrn3.png" alt="Loan Synthetic Data" width="1200"/>
